# PHY abstraction

> A transmission's packet error rate from its SINR, without its bits; and the draw of what arrives.

In [ ]:
#| default_exp phy_abstraction

In [ ]:
#| hide
from nbdev.showdoc import *

A *PHY abstraction* gives the packet error rate of a transmission from its SINR, without simulating its bits.
An `ErrorModel` does it for a batch of transmissions at once:

* `BPSKErrorModel` (the default) uses uncoded BPSK over AWGN. Its bit error rate is `Q(√(2 SINR))`, with
  independent bit errors, so bigger packets fail more often.
* `ThresholdErrorModel` receives a packet when the SINR is at or above a threshold.
* `EESMErrorModel` combines the subcarriers by EESM, then applies a logistic packet-error curve with your
  calibration, e.g. a Wi-Fi MCS.
* `SionnaErrorModel` (see `sionna_sys`) uses 5G NR with LDPC codes: Sionna SYS's calibrated tables, the
  subcarriers combined by EESM, and link adaptation.

The two simple models use the mean SINR over the subcarriers. The functions below are the analytic rates they use.

An `ErrorModel` is a block: called with the SINR of a step's transmissions at each subcarrier (`sinr`), their sizes
(`size_bits`), their broadcasts (`group`) and whether they are sent (`active`), it gives their packet error rates
(`per`), symbols on the air (`num_symbols`) and metadata (`info`). It rates only the transmissions sent and not
between nodes at the same place (an infinite SINR, lost never), so a new model only writes `rate`.

This is Sionna SYS's `sionna.sys.phy_abstraction` (`PHYAbstraction`), with its `sionna.sys.effective_sinr` (EESM).
Sionna's `PHYAbstraction` also draws which blocks are decoded; here `Decide` does, so a model only gives rates.

In [ ]:
#| export
from __future__ import annotations

import math
from abc import ABC, abstractmethod
from typing import List, Sequence, Tuple

import numpy as np

from comm_core.block import Block

In [ ]:
#| export
def bpsk_bit_error_rate(
    snr: float, # Linear signal-to-noise ratio
) -> float:
    "Bit error rate of BPSK over AWGN: Q(sqrt(2 snr))."
    return 0.5 * math.erfc(math.sqrt(max(snr, 0.0)))


def packet_error_rate(
    snr_db: float, # Signal-to-noise ratio (dB); -inf: no signal
    size_bits: int, # Packet size
    model: str = 'bpsk', # 'bpsk' (independent BPSK bit errors) or 'threshold'
    threshold_db: float = 10.0, # 'threshold': packets at or above this SNR are received
) -> float:
    "Probability that a packet is lost."
    if snr_db == math.inf:
        return 0.0
    if snr_db == -math.inf:
        return 1.0
    if model == 'threshold':
        return 0.0 if snr_db >= threshold_db else 1.0
    if model == 'bpsk':
        ber = bpsk_bit_error_rate(10 ** (snr_db / 10))
        return 1.0 - (1.0 - ber) ** max(int(size_bits), 1)
    raise ValueError(f"model: 'bpsk' or 'threshold', not {model!r}")

In [ ]:
assert packet_error_rate(30, 1000) < 1e-6 and packet_error_rate(-5, 1000) > 0.99
assert packet_error_rate(4, 10) < packet_error_rate(4, 1000)                  # bigger packets fail more often
assert packet_error_rate(9, 100, 'threshold') == 1.0 and packet_error_rate(10, 100, 'threshold') == 0.0
assert packet_error_rate(-math.inf, 8) == 1.0 and packet_error_rate(math.inf, 8) == 0.0

In [ ]:
#| export
class ErrorModel(Block, ABC):
    """A PHY abstraction: the packet error rate of transmissions, from their SINR. Subclasses write `rate`.

    The copies of a broadcast are one emission: a model that picks a rate (e.g. an MCS) gives them the same one."""

    outputs = ('per', 'num_symbols', 'info')

    def call(
        self,
        sinr: Sequence[np.ndarray], # Linear SINR of each transmission at each of its subcarriers, e.g. (n, K)
        size_bits: np.ndarray, # Packet size of each transmission
        group: Sequence | None = None, # The broadcast of each transmission, None for a unicast (None: all unicasts)
        active: np.ndarray | None = None, # Whether each one is sent (None: all)
    ) -> Tuple[np.ndarray, np.ndarray, List[dict]]:
        "(packet error rates, symbols on the air, metadata) of every transmission; `rate` gives them for the ones to rate."
        n = len(sinr)
        size = np.asarray(size_bits).astype(np.int64).reshape(n)
        per, symbols, info = np.zeros(n), size.copy(), [{} for _ in range(n)]
        rows = [k for k in range(n) if (active is None or active[k]) and not np.isinf(np.asarray(sinr[k])).any()]
        if rows:
            groups = None if group is None else [group[k] for k in rows]
            if groups is not None and all(g is None for g in groups):
                groups = None
            p, s, extra = self.rate([np.asarray(sinr[k], float).reshape(-1) for k in rows], np.maximum(size[rows], 1), groups)
            for k, pk, sk, ik in zip(rows, p, s, extra):
                per[k], symbols[k], info[k] = float(pk), int(sk), ik
        return per, symbols, info

    @abstractmethod
    def rate(
        self,
        sinr: List[np.ndarray], # Linear SINR of each transmission to rate, at each of its subcarriers (one value: flat)
        size_bits: np.ndarray, # Their packet sizes (at least 1 bit)
        group: Sequence | None = None, # Their broadcasts, None for a unicast (None: all unicasts)
    ) -> Tuple[np.ndarray, np.ndarray, List[dict]]:
        "(packet error rates, symbols on the air, extra metadata) of the transmissions."


def _mean_snr_db(sinr: np.ndarray) -> float:
    snr = float(np.mean(sinr))
    return 10 * math.log10(snr) if snr > 0 else -math.inf


class BPSKErrorModel(ErrorModel):
    "Uncoded BPSK over AWGN at the mean SINR (see `packet_error_rate`); one symbol per bit."

    def rate(self, sinr, size_bits, group=None):
        per = [packet_error_rate(_mean_snr_db(s), max(int(b), 1)) for s, b in zip(sinr, size_bits)]
        return np.array(per, float), np.asarray(size_bits, int), [{} for _ in sinr]


class ThresholdErrorModel(ErrorModel):
    "Received at or above an SINR (the mean over the subcarriers), lost below; one symbol per bit."

    def __init__(
        self,
        threshold_db: float = 10.0, # The SINR at which packets are received
    ):
        self.threshold_db = threshold_db

    def rate(self, sinr, size_bits, group=None):
        per = [packet_error_rate(_mean_snr_db(s), 1, 'threshold', self.threshold_db) for s in sinr]
        return np.array(per, float), np.asarray(size_bits, int), [{} for _ in sinr]

In [ ]:
show_doc(ErrorModel.call)

In [ ]:
show_doc(ErrorModel.rate)

In [ ]:
#| export
def eesm(
    sinr: np.ndarray, # Linear SINR at each subcarrier
    beta: float, # EESM parameter (of an MCS)
) -> float:
    "Effective SINR (linear) of SINRs at several subcarriers: -β ln(mean(exp(-SINR/β)))."
    x = -np.asarray(sinr, float) / beta
    m = x.max()
    return float(max(-beta * (m + math.log(np.mean(np.exp(x - m)))), 0.0))


class EESMErrorModel(ErrorModel):
    "EESM over the subcarriers, then a logistic packet error rate: a PHY abstraction calibrated by its parameters (e.g. a Wi-Fi MCS)."

    def __init__(
        self,
        beta: float, # EESM parameter of the MCS
        threshold_db: float, # Effective SINR at which a `reference_bytes` packet is lost 10% of the time
        slope: float = 1.5, # Steepness of the waterfall (1/dB)
        reference_bytes: int = 1000, # Packet size of `threshold_db`; other sizes scale as independent pieces
        bits_per_symbol: float = 1.0, # For the symbols on the air (the latency of `SystemLevelChannel`)
    ):
        self.beta, self.threshold_db, self.slope = beta, threshold_db, slope
        self.reference_bytes, self.bits_per_symbol = reference_bytes, bits_per_symbol

    def rate(self, sinr, size_bits, group=None):
        per, info = [], []
        for s, bits in zip(sinr, size_bits):
            eff = eesm(s, self.beta)
            eff_db = 10 * math.log10(eff) if eff > 0 else -math.inf
            if eff_db == -math.inf:
                p = 1.0
            else:
                p_ref = 1 / (1 + math.exp(min(self.slope * (eff_db - self.threshold_db) + math.log(9.0), 700)))
                p = 1 - (1 - p_ref) ** (max(int(bits), 1) / 8 / self.reference_bytes)
            per.append(p)
            info.append({'sinr_eff_db': eff_db})
        symbols = np.ceil(np.asarray(size_bits, float) / self.bits_per_symbol).astype(int)
        return np.array(per, float), symbols, info

In [ ]:
wifi = EESMErrorModel(beta=28.0, threshold_db=21.0, slope=1.5)              # ~802.11ac MCS 7 (approximate)
per, _, info = wifi([np.full(52, 10 ** 2.1), np.full(52, 10 ** 2.6)], np.array([8000, 8000]))
assert abs(per[0] - 0.1) < 1e-9 and per[1] < 0.01 and abs(info[0]['sinr_eff_db'] - 21) < 1e-9   # 10% at the threshold
rng = np.random.default_rng(0)
faded = 10 ** 2.1 * np.abs((rng.standard_normal(52) + 1j * rng.standard_normal(52)) / np.sqrt(2)) ** 2
assert eesm(faded, 28.0) < np.mean(faded)                                   # faded subcarriers count for more
assert wifi([np.zeros(52)], np.array([100]))[0][0] == 1.0
assert wifi([np.full(52, 10 ** 2.1)], np.array([16000]))[0][0] > per[0]     # bigger packets fail more

In [ ]:
per, symbols, _ = BPSKErrorModel()([np.array([10 ** 0.6]), np.array([10 ** 0.2, 10.0])], np.array([100, 100]))
assert np.allclose(per, [packet_error_rate(6, 100), packet_error_rate(10 * np.log10((10 ** 0.2 + 10) / 2), 100)])
assert symbols.tolist() == [100, 100]
# as a block: the SINRs of a step, (n, K); not rated: the ones not sent, and nodes at the same place (never lost)
per, symbols, info = BPSKErrorModel()(np.array([[1.0], [math.inf], [1.0]]), np.array([10, 10, 0]), active=np.array([True, True, False]))
assert per[0] > 0 and per[1] == per[2] == 0 and symbols.tolist() == [10, 10, 0] and info == [{}, {}, {}]
assert ThresholdErrorModel(10)([np.array([9.9]), np.array([10.0])], np.array([8, 8]))[0].tolist() == [1.0, 0.0]

## What arrives

`Decide` draws whether each packet arrives: it is lost with its packet error rate. It draws once per transmission
sent, from its own generator (`reset(seed)` makes it reproducible).

In [ ]:
#| export
class Decide(Block):
    "Whether each packet arrives: lost with its packet error rate, one draw per transmission sent."

    outputs = ('success',)

    def call(
        self,
        per: np.ndarray, # Packet error rate of each transmission
        active: np.ndarray | None = None, # Whether each one is sent (None: all)
    ) -> np.ndarray:
        success = np.zeros(len(per), dtype=bool)
        for k in range(len(per)):
            if active is None or active[k]:
                success[k] = self.rng.random() >= per[k]
        return success

In [ ]:
show_doc(Decide.call)

In [ ]:
d = Decide()
d.reset(seed=0); first = d(np.full(2000, 0.3))
assert abs(1 - first.mean() - 0.3) < 0.03                              # lost 30% of the time
d.reset(seed=0); assert (d(np.full(2000, 0.3)) == first).all()          # reproducible
assert d(np.array([0.0, 0.0]), active=np.array([True, False])).tolist() == [True, False]   # not sent: not arrived

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()